# M3Care imputation-quality evaluation — Deterioration

This notebook evaluates imputation of artificially masked, originally observed CXR embeddings for the deterioration pathway. **Model A = Without CL** and **Model B = With CL**. Each model's imputed representations (GCN outputs) are compared against the original target embeddings produced by **its own encoder**, not with embeddings from the other model.

The notebook reports **three main evaluation figures and two supplementary paired-observation figures**. The ED stay is the unit of analysis and a patient may contribute multiple stays. “Original” denotes the target embedding before masking. “Learned target” denotes the learned cross-modal target network's output. These are representation-level diagnostics, not image reconstruction or downstream clinical performance.

| Figure | Display | Main question |
|---|---|---|
| 1 | Absolute GCN imputation quality |  How closely does each model reconstruct each selected modality? |
| 2 | Paired CL effect | What is the mean within-stay change (With CL − Without CL)? |
| 3 | Learned-target alignment diagnostic | How do Learned target → original, GCN → learned target and GCN → original compare? |
| 4 | Paired ED-stay trajectories | How does each matched observation change? |
| 5 | Distribution of paired CL effects | How variable are within-stay changes? |

## 1. Evaluation protocol and matching checks

For each eligible ED stay, the target modality is observed and encoded **before masking**, then artificially hidden while available non-target context is retained. The GCN output (`gcn_x0`), refined output (`final_imputed`) and, when available, learned-target diagnostics are recorded at the **observation level**. All cosine similarities in the main figures use the **raw embedding space**.

The paired analyses match observations using `sample_id`. When IDs are evaluation-order fallbacks (`eval_order_...`), matching IDs alone is insufficient. The following input-alignment and eligibility-mask checks should pass before treating paired lines or differences as matched ED stays. They verify the inspected input fields and selection order. Unique ED-stay identifiers would be stronger evidence. Use the full intended evaluation split for final reporting (`BATCH_LIMIT=None`).

In [ ]:
from pathlib import Path
import os
import sys
import math
import json
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from torch.utils.data import DataLoader

PROJECT_ROOT = Path('/data/mm-healthfusion-m3care-hospitalisation')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
EVAL_SPLIT = 'test'
BATCH_LIMIT = None  # Set to a small integer for a quick test; None = full split.
OUT_DIR = PROJECT_ROOT / 'outputs' / 'm3care_imputation_quality'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Visual style follows the supplied presentation code.
RUN_COLORS = {
    'Model A': '#6D2E8E',
    'Model B': '#0E7C84',
}
CHAIN_COLORS = {
    'Learned target -> true': '#6D2E8E',
    'GCN -> learned target': '#0E7C84',
    'GCN -> true': '#BE7C1C',
}

plt.rcParams.update({
    'font.size': 9,
    'axes.titlesize': 10,
    'axes.labelsize': 9,
})

print('PROJECT_ROOT:', PROJECT_ROOT)
print('DEVICE:', DEVICE)
print('OUT_DIR:', OUT_DIR)

# ---------------- User settings ----------------
# MODEL_A_CHECKPOINT = Path('/path/to/model_A/checkpoint_best.ckpt')
# MODEL_A_CONFIG = Path('/path/to/model_A/config.yaml')
# MODEL_B_CHECKPOINT = Path('/path/to/model_B/checkpoint_best.ckpt')
# MODEL_B_CONFIG = Path('/path/to/model_B/config.yaml')
MODEL_A_CHECKPOINT = Path('/data/logs/ablation/m3care/deterioration/allStay/averageRecord/allCohort/normalised_ohe/deterioration_mechanical_ventilation/runs/m3care_m3care_stabloss_similarity_w0_01_imputedMiss_maskFusion_3_1ohe/checkpoint_best.ckpt')
MODEL_A_CONFIG = Path('/data/logs/ablation/m3care/deterioration/allStay/averageRecord/allCohort/normalised_ohe/deterioration_mechanical_ventilation/runs/m3care_m3care_stabloss_similarity_w0_01_imputedMiss_maskFusion_3_1ohe/config.yaml')
MODEL_B_CHECKPOINT = Path('/data/logs/ablation/m3care/deterioration/allStay/averageRecord/allCohort/normalised_ohe/deterioration_mechanical_ventilation/runs/m3care_m3care_deterBest_CL_finetune_bl_MV_3_1ohe/checkpoint_best.ckpt')
MODEL_B_CONFIG = Path('/data/logs/ablation/m3care/deterioration/allStay/averageRecord/allCohort/normalised_ohe/deterioration_mechanical_ventilation/runs/m3care_m3care_deterBest_CL_finetune_bl_MV_3_1ohe/config.yaml')


# Select the target modalities to include in the plots.
# For the current deterioration study, this should normally be ['cxr'].
TARGET_MODALITIES = ['cxr']
REQUIRE_ALL_CONTEXT_MODALITIES = False

print('TARGET_MODALITIES:', TARGET_MODALITIES)

In [ ]:
import yaml
from src.data.splits import get_dataloaders
from src.training.lightning import MMModel


def load_cfg(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'Config does not exist: {path}')
    if path.suffix.lower() not in {'.yaml', '.yml'}:
        raise ValueError(f'Expected YAML config, got: {path}')
    with path.open('r', encoding='utf-8') as f:
        return yaml.safe_load(f) or {}


def infer_feature_dims(batch, model_cfg):
    modality_names = list(model_cfg.get('modality_names', [])) or ['tabular', 'ecg', 'cxr']
    aliases = {
        'tabular': ['tab_feats', 'static_feats'],
        'static': ['static_feats', 'tab_feats'],
    }
    dims = {}
    for modality in modality_names:
        found = None
        for key in aliases.get(modality, [f'{modality}_feats']):
            if key in batch and torch.is_tensor(batch[key]):
                found = int(batch[key].shape[-1])
                break
        if found is None:
            raise KeyError(
                f'Cannot infer feature dimension for {modality!r}. '
                f'Available batch keys: {sorted(batch.keys())}'
            )
        dims[modality] = found
    if 'tabular' in dims:
        dims.setdefault('static', dims['tabular'])
    if 'static' in dims:
        dims.setdefault('tabular', dims['static'])
    return dims


def build_eval_loader(cfg, split):
    loaders = get_dataloaders(cfg)
    if split not in loaders:
        raise KeyError(f'Split {split!r} not available: {list(loaders)}')
    loader = loaders[split]
    return DataLoader(
        loader.dataset,
        batch_size=loader.batch_size,
        shuffle=False,
        num_workers=loader.num_workers,
        collate_fn=loader.collate_fn,
        pin_memory=loader.pin_memory,
        drop_last=False,
    )


def load_model(checkpoint_path, config_path, split):
    cfg = load_cfg(config_path)
    model_cfg = dict(cfg.get('model', {}))
    model_cfg.pop('load_from', None)
    fusion_method = str(model_cfg.get('fusion_method', model_cfg.get('name', 'm3care'))).lower()
    if fusion_method != 'm3care':
        raise ValueError(f'Expected M3Care, got {fusion_method!r}')

    loader = build_eval_loader(cfg, split)
    first_batch = next(iter(loader))
    feature_dims = infer_feature_dims(first_batch, model_cfg)

    lit_model = MMModel(
        model=None,
        learning_rate=float(cfg.get('train', {}).get('learning_rate', 3e-4)),
        weight_decay=float(cfg.get('train', {}).get('weight_decay', 1e-4)),
        pos_weight=None,
        scheduler_patience=None,
        scheduler_monitor='val_mcc',
        fusion_method=fusion_method,
        fusion_config=model_cfg,
        feature_dims=feature_dims,
        modality_dropout_config={},
    )
    lit_model.load_pretrained_weights(str(checkpoint_path), strict=True)
    lit_model.to(DEVICE)
    lit_model.eval()
    return lit_model, loader, cfg

In [ ]:
def move_batch_to_device(batch):
    return {k: (v.to(DEVICE) if torch.is_tensor(v) else v) for k, v in batch.items()}


def model_masks(model, batch, encoded):
    return model._modality_mask(batch, encoded)


def prefix_for_modality(modality):
    return 'tab' if modality == 'tabular' else modality


def mask_batch_modality(batch, modality, row_mask):
    edited = dict(batch)
    prefix = prefix_for_modality(modality)
    for suffix in ('missing', 'lengths', 'feats', 'time'):
        key = f'{prefix}_{suffix}'
        if key in edited and torch.is_tensor(edited[key]):
            edited[key] = edited[key].clone()
    for suffix, value in [('missing', 1), ('lengths', 0), ('feats', 0), ('time', 0)]:
        key = f'{prefix}_{suffix}'
        if key in edited:
            edited[key][row_mask] = value
    return edited


def context_modalities_for_model(model, target_modality):
    names = list(getattr(model.imputation, 'modality_names', []))
    return [m for m in names if m != target_modality]


def eligible_rows(masks, target_modality, model, require_all_context=False):
    if target_modality not in masks:
        raise KeyError(f'Target modality {target_modality!r} not found in masks: {list(masks)}')
    target_present = masks[target_modality].reshape(-1).bool()
    context_modalities = context_modalities_for_model(model, target_modality)
    context_available = [masks[m].reshape(-1).bool() for m in context_modalities if m in masks]
    if not context_available:
        return target_present, context_modalities
    stacked = torch.stack(context_available, dim=0)
    context_ok = stacked.all(dim=0) if require_all_context else stacked.any(dim=0)
    return target_present & context_ok, context_modalities


def _scalarize_id(value, i):
    if torch.is_tensor(value):
        v = value[i]
        if torch.is_tensor(v):
            v = v.detach().cpu().item() if v.numel() == 1 else tuple(v.detach().cpu().reshape(-1).tolist())
        return str(v)
    if isinstance(value, (list, tuple)):
        return str(value[i])
    return str(value)


def sample_ids_from_batch(batch, start_index, batch_size):
    """Use dataset identifiers when available; otherwise fall back to stable evaluation order."""
    preferred = [
        'sample_id', 'patient_id', 'subject_id', 'stay_id', 'hadm_id',
        'icustay_id', 'study_id', 'row_id', 'index',
    ]
    for key in preferred:
        if key in batch:
            value = batch[key]
            try:
                ids = [_scalarize_id(value, i) for i in range(batch_size)]
                if len(set(ids)) == len(ids):
                    return ids
            except Exception:
                pass
    return [f'eval_order_{start_index + i:08d}' for i in range(batch_size)]


def _gcn_impute_selected_target(model, masked_encoded, masked_masks, target_modality):
    imp = model.imputation
    if target_modality not in imp.modalities_to_impute:
        raise ValueError(
            f'{target_modality!r} is not in modalities_to_impute: {imp.modalities_to_impute}'
        )
    sim_list, mask_list = [], []
    for idx, name in enumerate(imp.modality_names):
        x = masked_encoded[name]
        mask = masked_masks[name]
        mask2 = mask * mask.transpose(0, 1)
        sim, _ = imp._build_similarity(x, imp.simiProj[idx], imp.eps_params[idx])
        sim = sim * mask2
        sim_list.append(sim)
        mask_list.append(mask2)
    sum_masks = torch.stack(mask_list, dim=0).sum(dim=0).clamp_min(1.0)
    similar_score = torch.stack(sim_list, dim=0).sum(dim=0) / sum_masks
    threshold = imp.sigmoid(imp.threshold).reshape(-1)[0]
    similar_score = imp.relu(similar_score - threshold)
    bin_mask = similar_score > 0
    similar_score = similar_score + bin_mask * threshold.detach()
    target_idx = imp.modalities_to_impute.index(target_modality)
    return imp._refine_with_gcn(
        similar_score,
        masked_encoded[target_modality],
        masked_masks[target_modality],
        imp.GCN[target_idx],
        imp.GCN_2[target_idx],
        imp.weight1[target_idx],
        imp.weight2[target_idx],
    )[:2]


def raw_metric_vectors(pred, true):
    pred, true = pred.float(), true.float()
    cosine = F.cosine_similarity(pred, true, dim=-1, eps=1e-8)
    mse = ((pred - true) ** 2).mean(dim=-1)
    mae = (pred - true).abs().mean(dim=-1)
    true_norm = true.norm(dim=-1)
    pred_norm = pred.norm(dim=-1)
    rel_norm = (pred_norm - true_norm).abs() / true_norm.clamp_min(1e-8)
    return cosine, mse, mae, rel_norm


def projected_pair(anchor, target, imp):
    anchor_z = F.normalize(imp.contrastive_projector(anchor), p=2, dim=-1, eps=1e-8)
    target_z = F.normalize(imp.contrastive_target_projector(target.detach()), p=2, dim=-1, eps=1e-8)
    return anchor_z, target_z


def projected_diagnostics(anchor_z, target_z, temperature):
    n = anchor_z.size(0)
    sim = anchor_z @ target_z.T
    labels = torch.arange(n, device=sim.device)
    logits = sim / temperature
    row_ce = F.cross_entropy(logits, labels, reduction='none')
    col_ce = F.cross_entropy(logits.T, labels, reduction='none')
    nce = 0.5 * (row_ce + col_ce)
    retrieval = (sim.argmax(dim=1) == labels).float()
    if n > 1:
        neg_mask = ~torch.eye(n, dtype=torch.bool, device=sim.device)
        neg = sim[neg_mask].reshape(n, n - 1)
        gap = sim.diag() - neg.mean(dim=1)
    else:
        gap = torch.full((n,), float('nan'), device=sim.device)
    return sim.diag(), retrieval, gap, nce


def learned_target_available(model):
    imp = model.imputation
    return (
        bool(getattr(imp, 'learned_cross_modal_target', False))
        and hasattr(imp, 'learned_target_networks')
        and hasattr(imp, 'contrastive_target_projector')
        and hasattr(imp, 'contrastive_projector')
    )


def learned_target_for_batch(model, encoded, masks, target_modality):
    if not learned_target_available(model):
        return None
    target, _ = model.imputation._build_cross_modal_target(encoded, masks, target_modality)
    return target.detach()


def evaluate_one_batch(lit_model, batch, target_modality, start_index, require_all_context=False):
    model = lit_model.model
    with torch.no_grad():
        original_batch = move_batch_to_device(batch)
        original_encoded = model.encode(original_batch)
        original_masks = model_masks(model, original_batch, original_encoded)
        eligible, context_modalities = eligible_rows(original_masks, target_modality, model, require_all_context)
        if not bool(eligible.any().item()):
            return None, 0

        n_batch = int(eligible.sum().item())
        true_target = original_encoded[target_modality][eligible].detach()
        sample_ids_all = sample_ids_from_batch(original_batch, start_index, len(eligible))
        sample_ids = [sample_ids_all[i] for i, ok in enumerate(eligible.detach().cpu().tolist()) if ok]

        masked_batch = mask_batch_modality(original_batch, target_modality, eligible)
        masked_encoded = model.encode(masked_batch)
        masked_masks = model_masks(model, masked_batch, masked_encoded)
        x0_all, x_final_all = _gcn_impute_selected_target(model, masked_encoded, masked_masks, target_modality)
        x0 = x0_all[eligible].detach()
        x_final = x_final_all[eligible].detach()
        teacher_all = learned_target_for_batch(model, masked_encoded, masked_masks, target_modality)
        teacher = None if teacher_all is None else teacher_all[eligible].detach()

    return {
        'sample_id': sample_ids,
        'true_target': true_target,
        'x0': x0,
        'x_final': x_final,
        'teacher': teacher,
    }, n_batch


def evaluate_model(lit_model, loader, model_name, target_modality, require_all_context=False, batch_limit=None):
    rows = []
    processed_batches = 0
    total_eligible = 0
    imp = lit_model.model.imputation
    teacher_enabled = learned_target_available(lit_model.model)
    start_index = 0

    for batch in loader:
        if batch_limit is not None and processed_batches >= batch_limit:
            break
        batch_size = next(v.shape[0] for v in batch.values() if torch.is_tensor(v) and v.ndim > 0)
        result, n_batch = evaluate_one_batch(
            lit_model, batch, target_modality, start_index,
            require_all_context=require_all_context,
        )
        start_index += batch_size
        processed_batches += 1
        if result is None:
            continue
        total_eligible += n_batch

        true_target = result['true_target']
        x0 = result['x0']
        x_final = result['x_final']
        ids = result['sample_id']

        for rep_name, pred in [('gcn_x0', x0), ('final_imputed', x_final)]:
            cosine, mse, mae, rel_norm = raw_metric_vectors(pred, true_target)
            for i, sid in enumerate(ids):
                rows.append({
                    'sample_id': sid,
                    'model': model_name,
                    'representation': rep_name,
                    'gcn_vs_true_raw_cosine': float(cosine[i].cpu()),
                    'gcn_vs_true_raw_mse': float(mse[i].cpu()),
                    'gcn_vs_true_raw_mae': float(mae[i].cpu()),
                    'gcn_vs_true_raw_relative_norm_error': float(rel_norm[i].cpu()),
                    'teacher_available': teacher_enabled,
                })

        teacher = result['teacher']
        if teacher is None or x0.shape[0] < 2:
            continue

        teacher_cos, teacher_mse, teacher_mae, teacher_rel = raw_metric_vectors(teacher, true_target)
        gct_cos, gct_mse, gct_mae, gct_rel = raw_metric_vectors(x0, teacher)

        teacher_z, true_z = projected_pair(teacher, true_target, imp)
        gct_z, teacher_t_z = projected_pair(x0, teacher, imp)
        gct_true_z, true_target_z = projected_pair(x0, true_target, imp)

        t_proj_cos, t_retr, t_gap, t_nce = projected_diagnostics(
            teacher_z, true_z,
            float(getattr(imp, 'learned_target_contrastive_temperature', imp.contrastive_temperature)),
        )
        gt_proj_cos, gt_retr, gt_gap, gt_nce = projected_diagnostics(
            gct_z, teacher_t_z, float(imp.contrastive_temperature)
        )
        gtrue_proj_cos, gtrue_retr, gtrue_gap, _ = projected_diagnostics(
            gct_true_z, true_target_z, float(imp.contrastive_temperature)
        )

        for i, sid in enumerate(ids):
            rows.append({
                'sample_id': sid,
                'model': model_name,
                'representation': 'teacher_chain',
                'teacher_vs_true_raw_cosine': float(teacher_cos[i].cpu()),
                'teacher_vs_true_raw_mse': float(teacher_mse[i].cpu()),
                'teacher_vs_true_raw_mae': float(teacher_mae[i].cpu()),
                'teacher_vs_true_raw_relative_norm_error': float(teacher_rel[i].cpu()),
                'teacher_vs_true_projected_cosine': float(t_proj_cos[i].cpu()),
                'teacher_vs_true_retrieval_accuracy': float(t_retr[i].cpu()),
                'teacher_vs_true_pos_neg_gap': float(t_gap[i].cpu()),
                'teacher_target_infonce': float(t_nce[i].cpu()),
                'gcn_vs_teacher_raw_cosine': float(gct_cos[i].cpu()),
                'gcn_vs_teacher_raw_mse': float(gct_mse[i].cpu()),
                'gcn_vs_teacher_raw_mae': float(gct_mae[i].cpu()),
                'gcn_vs_teacher_raw_relative_norm_error': float(gct_rel[i].cpu()),
                'gcn_vs_teacher_projected_cosine': float(gt_proj_cos[i].cpu()),
                'gcn_vs_teacher_retrieval_accuracy': float(gt_retr[i].cpu()),
                'gcn_vs_teacher_pos_neg_gap': float(gt_gap[i].cpu()),
                'gcn_to_teacher_infonce': float(gt_nce[i].cpu()),
                'gcn_vs_true_projected_cosine': float(gtrue_proj_cos[i].cpu()),
                'gcn_vs_true_retrieval_accuracy': float(gtrue_retr[i].cpu()),
                'gcn_vs_true_pos_neg_gap': float(gtrue_gap[i].cpu()),
            })

    return pd.DataFrame(rows), processed_batches, total_eligible


def run_one_window(window, target_modalities, model_paths, split='val', require_all_context=False, batch_limit=None):
    paths = model_paths[window]
    model_a, loader_a, _ = load_model(paths['model_A']['checkpoint'], paths['model_A']['config'], split)
    model_b, loader_b, _ = load_model(paths['model_B']['checkpoint'], paths['model_B']['config'], split)
    chunks = []
    for target_modality in target_modalities:
        for label, model, loader in [('Model A', model_a, loader_a), ('Model B', model_b, loader_b)]:
            if target_modality not in model.model.imputation.modalities_to_impute:
                raise ValueError(
                    f'{target_modality!r} is not imputable by {label} in {window}: '
                    f'{model.model.imputation.modalities_to_impute}'
                )
        ra, ba, na = evaluate_model(model_a, loader_a, 'Model A', target_modality, require_all_context, batch_limit)
        rb, bb, nb = evaluate_model(model_b, loader_b, 'Model B', target_modality, require_all_context, batch_limit)
        r = pd.concat([ra, rb], ignore_index=True)
        r.insert(0, 'window', window)
        r.insert(1, 'target_modality', target_modality)
        chunks.append(r)
        print(f'{window} / {target_modality}: Model A n={na}, Model B n={nb}')
    return pd.concat(chunks, ignore_index=True) if chunks else pd.DataFrame()

## 2. Run the evaluation

Load both models and evaluate the eligible, originally observed targets using artificial masking. The next cells also verify input and eligibility alignment before interpreting paired results.

In [ ]:
model_a, loader_a, cfg_a = load_model(MODEL_A_CHECKPOINT, MODEL_A_CONFIG, EVAL_SPLIT)
model_b, loader_b, cfg_b = load_model(MODEL_B_CHECKPOINT, MODEL_B_CONFIG, EVAL_SPLIT)

all_chunks = []
for target_modality in TARGET_MODALITIES:
    for label, model in [('Model A', model_a), ('Model B', model_b)]:
        if target_modality not in model.model.imputation.modalities_to_impute:
            raise ValueError(
                f'{target_modality!r} is not imputable by {label}: '
                f'{model.model.imputation.modalities_to_impute}'
            )
    ra, ba, na = evaluate_model(model_a, loader_a, 'Model A', target_modality,
                                REQUIRE_ALL_CONTEXT_MODALITIES, BATCH_LIMIT)
    rb, bb, nb = evaluate_model(model_b, loader_b, 'Model B', target_modality,
                                REQUIRE_ALL_CONTEXT_MODALITIES, BATCH_LIMIT)
    r = pd.concat([ra, rb], ignore_index=True)
    r.insert(0, 'target_modality', target_modality)
    all_chunks.append(r)
    print(f'{target_modality}: Model A n={na}; Model B n={nb}')

results = pd.concat(all_chunks, ignore_index=True) if all_chunks else pd.DataFrame()
print('Total patient-level rows:', len(results))

# results.head() # Commented out to avoid data leakage

### Verify observation alignment

The following checks compare the two DataLoaders and their eligibility masks. Both should pass before interpreting evaluation-order fallback IDs as matched ED stays.

In [ ]:
def check_dataloader_alignment(
    loader_a,
    loader_b,
    atol=1e-6,
    rtol=1e-5,
):
    """
    Check whether two evaluation DataLoaders return identical
    observations in the same order.

    Compares raw inputs, not model-generated embeddings.
    """

    if len(loader_a.dataset) != len(loader_b.dataset):
        print(
            "WARNING: Different dataset sizes:",
            len(loader_a.dataset),
            len(loader_b.dataset),
        )
        return False

    print("Dataset size:", len(loader_a.dataset))

    # Compare features and missingness indicators.
    keys_to_check = [
        "tab_feats",
        "ecg_feats",
        "cxr_feats",
        "tab_missing",
        "ecg_missing",
        "cxr_missing",
    ]

    total_samples = 0
    mismatch_counts = {key: 0 for key in keys_to_check}
    compared_counts = {key: 0 for key in keys_to_check}

    if len(loader_a) != len(loader_b):
        print("WARNING: Different number of batches.")
        return False

    for batch_a, batch_b in zip(loader_a, loader_b):

        # Determine batch size.
        batch_size_a = next(
            v.shape[0]
            for v in batch_a.values()
            if torch.is_tensor(v) and v.ndim > 0
        )

        batch_size_b = next(
            v.shape[0]
            for v in batch_b.values()
            if torch.is_tensor(v) and v.ndim > 0
        )

        if batch_size_a != batch_size_b:
            print("WARNING: Different batch sizes.")
            return False

        total_samples += batch_size_a

        for key in keys_to_check:

            if key not in batch_a or key not in batch_b:
                continue

            a = batch_a[key].detach().cpu()
            b = batch_b[key].detach().cpu()

            if a.shape != b.shape:
                mismatch_counts[key] += batch_size_a
                continue

            if a.dtype.is_floating_point:
                equal = torch.isclose(
                    a,
                    b,
                    atol=atol,
                    rtol=rtol,
                    equal_nan=True,
                )
            else:
                equal = a == b

            # Reduce all non-batch dimensions.
            equal_per_sample = equal.reshape(
                batch_size_a, -1
            ).all(dim=1)

            mismatch_counts[key] += int(
                (~equal_per_sample).sum().item()
            )

            compared_counts[key] += batch_size_a

    print("\n=== DataLoader alignment ===")
    print("Samples checked:", total_samples)

    all_aligned = True

    for key in keys_to_check:
        print(
            f"{key}: "
            f"{mismatch_counts[key]} mismatched / "
            f"{compared_counts[key]} compared"
        )

        if mismatch_counts[key] > 0:
            all_aligned = False

        if compared_counts[key] != total_samples:
            print(f"  WARNING: {key} was not checked for every sample.")
            all_aligned = False

    if all_aligned:
        print("\nPASS: Input observations align in evaluation order.")
    else:
        print("\nFAIL: Alignment was not established.")

    return all_aligned

In [ ]:
aligned = check_dataloader_alignment(
    loader_a,
    loader_b,
)

In [ ]:
def check_eligible_alignment(
    model_a,
    model_b,
    loader_a,
    loader_b,
    target_modality="cxr",
    require_all_context=False,
):
    """
    Verify that the two models select the same eligible
    observations in the same evaluation order.
    """

    eligible_a_all = []
    eligible_b_all = []

    for batch_a, batch_b in zip(loader_a, loader_b):

        with torch.no_grad():

            batch_a_device = move_batch_to_device(batch_a)
            batch_b_device = move_batch_to_device(batch_b)

            inner_a = model_a.model
            inner_b = model_b.model

            encoded_a = inner_a.encode(batch_a_device)
            encoded_b = inner_b.encode(batch_b_device)

            masks_a = model_masks(
                inner_a, batch_a_device, encoded_a
            )
            masks_b = model_masks(
                inner_b, batch_b_device, encoded_b
            )

            eligible_a, _ = eligible_rows(
                masks_a,
                target_modality,
                inner_a,
                require_all_context=require_all_context,
            )

            eligible_b, _ = eligible_rows(
                masks_b,
                target_modality,
                inner_b,
                require_all_context=require_all_context,
            )

            eligible_a_all.append(
                eligible_a.detach().cpu()
            )
            eligible_b_all.append(
                eligible_b.detach().cpu()
            )

    eligible_a_all = torch.cat(eligible_a_all)
    eligible_b_all = torch.cat(eligible_b_all)

    same = torch.equal(
        eligible_a_all,
        eligible_b_all,
    )

    print("=== Eligible observation alignment ===")
    print("Model A eligible:", int(eligible_a_all.sum()))
    print("Model B eligible:", int(eligible_b_all.sum()))
    print(
        "Different eligibility decisions:",
        int((eligible_a_all != eligible_b_all).sum()),
    )
    print("Identical eligibility masks:", same)

    return same

In [ ]:
eligible_aligned = check_eligible_alignment(
    model_a,
    model_b,
    loader_a,
    loader_b,
    target_modality="cxr",
    require_all_context=REQUIRE_ALL_CONTEXT_MODALITIES,
)

In [ ]:
assert aligned, "Model A and B inputs are not aligned."
assert eligible_aligned, "Eligible observations are not aligned."

print("Pairing checks passed.")

In [ ]:
def check_pairing(results):
    gcn = results[
        results["representation"] == "gcn_x0"
    ].copy()

    group_cols = (
        ["window", "target_modality"]
        if "window" in gcn.columns
        else ["target_modality"]
    )

    for key, group in gcn.groupby(group_cols):
        a = group[group["model"] == "Model A"]
        b = group[group["model"] == "Model B"]

        ids_a = set(a["sample_id"])
        ids_b = set(b["sample_id"])

        print(f"\nConfiguration: {key}")
        print(f"Model A observations: {len(a)}")
        print(f"Model B observations: {len(b)}")
        print(f"Matched unique IDs: {len(ids_a & ids_b)}")
        print(f"Only Model A: {len(ids_a - ids_b)}")
        print(f"Only Model B: {len(ids_b - ids_a)}")
        print(f"Duplicate IDs in A: {a['sample_id'].duplicated().sum()}")
        print(f"Duplicate IDs in B: {b['sample_id'].duplicated().sum()}")

        fallback = any(
            str(s).startswith("eval_order_")
            for s in ids_a | ids_b
        )
        print(f"Uses evaluation-order fallback: {fallback}")

check_pairing(results)

## 3. Summary tables

The evaluation records one row per eligible observation per representation. Summary means and approximate 95% confidence intervals are calculated from observation-level values, not from unweighted averages of batch means. The primary model means are calculated separately. The paired CL-effect interval is calculated from **within-observation differences**. The two interval types answer different questions.

In [ ]:
def mean_ci(values):
    values = pd.Series(values).dropna().astype(float)
    if len(values) == 0:
        return np.nan, np.nan, np.nan, 0
    mean = float(values.mean())
    if len(values) > 1:
        se = float(values.std(ddof=1)) / math.sqrt(len(values))
        lo, hi = mean - 1.96 * se, mean + 1.96 * se
    else:
        lo = hi = np.nan
    return mean, lo, hi, len(values)


def paired_ci(values):
    return mean_ci(values)


def primary_summary(results):
    rows = []
    for keys, grp in results.groupby([c for c in ['window', 'target_modality', 'model'] if c in results.columns]):
        if not isinstance(keys, tuple):
            keys = (keys,)
        key_dict = dict(zip([c for c in ['window', 'target_modality', 'model'] if c in results.columns], keys))
        gcn = grp[grp['representation'] == 'gcn_x0']
        m, lo, hi, n = mean_ci(gcn['gcn_vs_true_raw_cosine'])
        mse_m, _, _, _ = mean_ci(gcn['gcn_vs_true_raw_mse'])
        mae_m, _, _, _ = mean_ci(gcn['gcn_vs_true_raw_mae'])
        rn_m, _, _, _ = mean_ci(gcn['gcn_vs_true_raw_relative_norm_error'])
        rows.append({**key_dict, 'n': n, 'cosine_mean': m, 'cosine_ci_low': lo, 'cosine_ci_high': hi,
                     'mse_mean': mse_m, 'mae_mean': mae_m, 'relative_norm_error_mean': rn_m})
    return pd.DataFrame(rows)


def teacher_summary(results):
    rows = []
    if 'teacher_chain' not in set(results.get('representation', [])):
        return pd.DataFrame()
    for keys, grp in results[results['representation'] == 'teacher_chain'].groupby([c for c in ['window', 'target_modality', 'model'] if c in results.columns]):
        if not isinstance(keys, tuple): keys = (keys,)
        key_dict = dict(zip([c for c in ['window','target_modality','model'] if c in results.columns], keys))
        row = dict(key_dict)
        for prefix, cols in [
            ('teacher_vs_true', ['teacher_vs_true_raw_cosine','teacher_vs_true_raw_mse','teacher_vs_true_raw_mae','teacher_vs_true_projected_cosine','teacher_vs_true_retrieval_accuracy','teacher_target_infonce']),
            ('gcn_vs_teacher', ['gcn_vs_teacher_raw_cosine','gcn_vs_teacher_raw_mse','gcn_vs_teacher_raw_mae','gcn_vs_teacher_projected_cosine','gcn_vs_teacher_retrieval_accuracy','gcn_to_teacher_infonce']),
            ('gcn_vs_true_projected', ['gcn_vs_true_projected_cosine','gcn_vs_true_retrieval_accuracy']),
        ]:
            for c in cols:
                m, lo, hi, n = mean_ci(grp[c])
                row[f'{c}_mean'] = m; row[f'{c}_ci_low'] = lo; row[f'{c}_ci_high'] = hi
        row['n'] = int(len(grp))
        rows.append(row)
    return pd.DataFrame(rows)


def paired_effect_summary(results):
    rows = []
    base_cols = ['window','target_modality'] if 'window' in results.columns else ['target_modality']
    for keys, grp in results.groupby(base_cols):
        if not isinstance(keys, tuple): keys = (keys,)
        key_dict = dict(zip(base_cols, keys))
        a = grp[(grp['model']=='Model A') & (grp['representation']=='gcn_x0')][['sample_id','gcn_vs_true_raw_cosine']].rename(columns={'gcn_vs_true_raw_cosine':'cos_a'})
        b = grp[(grp['model']=='Model B') & (grp['representation']=='gcn_x0')][['sample_id','gcn_vs_true_raw_cosine']].rename(columns={'gcn_vs_true_raw_cosine':'cos_b'})
        # Pair observations within the same window/modality; reject duplicate IDs.
        # Evaluation-order IDs require identical loader ordering and eligibility.
        merged = a.merge(b, on='sample_id', how='inner', validate='one_to_one')
        delta = merged['cos_b'] - merged['cos_a']
        mean, lo, hi, n = paired_ci(delta)
        rows.append({**key_dict, 'n_paired': n, 'delta_cosine_mean': mean, 'delta_cosine_ci_low': lo, 'delta_cosine_ci_high': hi})
    return pd.DataFrame(rows)

In [ ]:
primary = primary_summary(results)
effect = paired_effect_summary(results)
teacher = teacher_summary(results)

print('=== Primary imputation quality ===')
display(primary)

print('=== Paired CL effect (Model B - Model A) ===')
display(effect)

print('=== Learned-target teacher chain ===')
display(teacher)

### How to read the accompanying tables

| Field | Interpretation |
|---|---|
| `n` / `n_paired` | Number of eligible observations for a model / number of matched pairs for the CL-effect analysis. |
| `cosine_mean`, `cosine_ci_low`, `cosine_ci_high` | Mean raw GCN → original cosine similarity and approximate 95% CI for each model; **higher** means stronger directional agreement. |
| `mse_mean`, `mae_mean` | Mean coordinate-wise squared / absolute embedding error; **lower** is better, but values depend on embedding scale. |
| `relative_norm_error_mean` | Relative difference in embedding magnitudes; **lower** is better. |
| `delta_cosine_mean`, `delta_cosine_ci_low`, `delta_cosine_ci_high` | Mean **With CL − Without CL** cosine difference and approximate 95% CI computed from matched differences; positive favours CL. |
| `teacher_vs_true_*` | **Learned target → original** metrics (the code retains `teacher` in internal field names). |
| `gcn_vs_teacher_*` | **GCN → learned target** metrics (internal field names retain `teacher`). |
| `gcn_vs_true_projected_*` | Similarity after contrastive projection; **not** the raw GCN → original reconstruction metric. |

The learned-target diagnostics are available only where the corresponding network is enabled. A CI crossing zero means the estimated paired effect is not clearly separated from zero under the reported interval method; it does not prove the absence of an effect.

In [ ]:
# Consistency check: the raw GCN->true metric for Model B must be the
# same in the primary table and in the orange teacher-chain bar.
# Graph 3 retrieves this value from primary_summary(results).
check_primary = primary_summary(results)
check_teacher = teacher_summary(results)
join_cols = ['target_modality']
if 'window' in results.columns:
    join_cols.insert(0, 'window')
if not check_teacher.empty:
    cl_primary = check_primary[check_primary['model'] == 'Model B']
    cl_teacher = check_teacher[check_teacher['model'] == 'Model B']
    matched_configs = cl_teacher[join_cols].merge(
        cl_primary[join_cols + ['cosine_mean']],
        on=join_cols, how='left', validate='one_to_one',
    )
    assert matched_configs['cosine_mean'].notna().all(), (
        'Some teacher-chain configurations lack primary GCN->true results.'
    )
    print('Raw GCN->true reference values for Graph 3 (must match Graph 1 With CL):')
    display(matched_configs)
else:
    print('No teacher-chain rows: teacher diagnostic unavailable.')

## 4. Evaluation figures

All figures retain the notebook's existing palette, confidence-interval conventions, transparent backgrounds and export settings. In model-comparison plots, purple means **Without CL** and teal means **With CL**. In Figure 3, colours identify **relationships**, not separate model runs.

### Main figures (1–3)

**Figure 1 — Absolute GCN imputation quality.** Each bar shows a model-specific mean of raw GCN → original cosine similarity over eligible stays with approximate 95% CIs. The interval is not a paired-difference interval.

**Figure 2 — Paired effect of CL.** Each bar shows the mean matched-observation difference **With CL − Without CL** in raw GCN → original cosine similarity, with an approximate 95% CI of the paired differences. Positive favours CL; negative favours Without CL. Pairing is justified only after observation alignment is verified.

**Figure 3 — Learned-target alignment diagnostic.** Purple = **Learned target → original** (quality of the learned target); teal = **GCN → learned target** (alignment with that target); orange = **GCN → original** (actual imputation alignment). All three use **raw embedding-space cosine similarities**. The orange value should agree with the With-CL result in Figure 1 for the same eligible subset. A strong learned target does not by itself demonstrate successful transfer to the GCN.

### Supplementary figures (4–5)

**Figure 4 — Paired ED-stay trajectories.** Each faint grey line connects the same matched ED stay under Without CL and With CL. Upward lines indicate improved cosine similarity with CL; downward lines indicate reduced similarity. Coloured points and error bars are the two **model-specific means and CIs on the matched subset**, not the CI for their difference. The title's `n` is the number of matched stays.

**Figure 5 — Distribution of paired CL effects.** Each histogram value is an individual matched stay's **With CL − Without CL** cosine difference. The zero line indicates no change and the second reference line marks the mean. The distribution shows heterogeneity and possible outliers, **not** uncertainty around the mean. Figure 2 reports the paired mean and its CI.

In [ ]:
MODEL_LABELS = {
    "Model A": "Without CL",
    "Model B": "With CL",
}
def _savefig(fig, filename):
    path = OUT_DIR / filename
    fig.savefig(path, dpi=600, transparent=True, bbox_inches='tight')
    print('Saved:', path)


def _bar_value_label(ax, x, value, upper_error=0.0):
    if pd.notna(value):
        ax.text(x, value + upper_error + 0.012, f'{value:.3f}', ha='center', va='bottom', fontsize=8, color='#404040')


def plot_primary_imputation(results, title='GCN imputation quality across modalities', filename='plot1_gcn_vs_true.png'):
    sm = primary_summary(results)
    if sm.empty:
        print('No primary results to plot.'); return
    configs = sm[['window','target_modality']].drop_duplicates().itertuples(index=False, name=None) if 'window' in sm.columns else sm[['target_modality']].drop_duplicates().itertuples(index=False, name=None)
    labels = [f'{w}\n{t}' for w,t in configs] if 'window' in sm.columns else [t for (t,) in configs]
    keys = list(sm[['window','target_modality']].drop_duplicates().itertuples(index=False, name=None)) if 'window' in sm.columns else [(t,) for t in sm['target_modality'].drop_duplicates()]
    x = np.arange(len(keys))
    runs = ['Model A','Model B']
    width = min(0.36, 0.72 / len(runs))
    fig, ax = plt.subplots(figsize=(5, 3.4), dpi=600)
    fig.patch.set_alpha(0.0); ax.patch.set_alpha(0.0)
    for ri, run in enumerate(runs):
        for i, key in enumerate(keys):
            q = sm[(sm['model']==run)]
            if 'window' in sm.columns:
                q = q[(q['window']==key[0]) & (q['target_modality']==key[1])]
            else:
                q = q[q['target_modality']==key[0]]
            if q.empty: continue
            r = q.iloc[0]; p=float(r['cosine_mean']); lo=p-float(r['cosine_ci_low']); hi=float(r['cosine_ci_high'])-p
            offset=(ri-(len(runs)-1)/2)*width
            ax.bar(x[i]+offset,p,width*0.92,color=RUN_COLORS[run],edgecolor='white',linewidth=0.5,
                   yerr=[[lo],[hi]],capsize=2,error_kw=dict(ecolor='lightgray',elinewidth=1),alpha=0.9)
            _bar_value_label(ax,x[i]+offset,p,hi)
    ax.set_xticks(x); ax.set_xticklabels(labels,fontsize=8)
    ax.tick_params(axis='x',length=0); ax.set_ylabel('Cosine similarity'); ax.set_title(title)
    ax.set_ylim(min(-0.02, float(sm['cosine_ci_low'].min())-0.05), min(1.08, float(sm['cosine_ci_high'].max())+0.10))

    ax.legend(
        handles=[
            Patch(
                facecolor=RUN_COLORS[r],
                edgecolor="white",
                label=MODEL_LABELS[r],
            )
            for r in runs
        ],
        fontsize=8,
        loc="upper center",
        bbox_to_anchor=(0.5, -0.20),
        frameon=False,
        ncol=2,
        handletextpad=0.6,
        borderaxespad=0.0,
    )
    plt.tight_layout(); _savefig(fig,filename); plt.show()


def plot_paired_cl_effect(results, title='Change in GCN imputation quality with CL', filename='plot2_cl_effect.png'):
    sm = paired_effect_summary(results)
    if sm.empty:
        print('No paired Model A/Model B results to plot.'); return
    labels = [f'{w}\n{t}' for w,t in sm[['window','target_modality']].itertuples(index=False, name=None)] if 'window' in sm.columns else sm['target_modality'].tolist()
    x = np.arange(len(sm))
    fig, ax = plt.subplots(figsize=(5, 3.4), dpi=600)
    fig.patch.set_alpha(0.0); ax.patch.set_alpha(0.0)
    for i, r in sm.iterrows():
        p=float(r['delta_cosine_mean']); lo=p-float(r['delta_cosine_ci_low']); hi=float(r['delta_cosine_ci_high'])-p
        ax.bar(x[i],p,0.60,color=RUN_COLORS['Model B'],edgecolor='white',linewidth=0.5,
               yerr=[[lo],[hi]],capsize=2,error_kw=dict(ecolor='lightgray',elinewidth=1),alpha=0.9)
        if p >= 0: _bar_value_label(ax,x[i],p,hi)
        else: ax.text(x[i], p-lo-0.012, f'{p:.3f}', ha='center', va='top', fontsize=8, color='#404040')
    ax.axhline(0,color='#404040',linewidth=0.8)
    ax.set_xticks(x); ax.set_xticklabels(labels,fontsize=8); ax.tick_params(axis='x',length=0)
    ax.set_ylabel('CL - no CL: Δ cosine similarity'); ax.set_title(title)
    ymax=max(abs(float(sm['delta_cosine_ci_low'].min())),abs(float(sm['delta_cosine_ci_high'].max()))) if len(sm) else 0.1
    ax.set_ylim(-max(0.05,ymax*1.35), max(0.05,ymax*1.35))
    plt.tight_layout(); _savefig(fig,filename); plt.show()


def plot_teacher_chain(results, title='Learned-target teacher-chain diagnostic', filename='plot3_teacher_chain.png', model='Model B'):
    teacher_sm = teacher_summary(results)
    gcn_sm = primary_summary(results)

    if teacher_sm.empty or gcn_sm.empty:
        print("Required teacher or GCN results are unavailable.")
        return

    teacher_sm = teacher_sm[teacher_sm["model"] == model].copy()
    gcn_sm = gcn_sm[gcn_sm["model"] == model].copy()

    merge_keys = ["model", "target_modality"]

    gcn_plot = gcn_sm[
        merge_keys + [
            "cosine_mean",
            "cosine_ci_low",
            "cosine_ci_high",
        ]
    ].rename(columns={
        "cosine_mean": "gcn_vs_true_raw_cosine_mean",
        "cosine_ci_low": "gcn_vs_true_raw_cosine_ci_low",
        "cosine_ci_high": "gcn_vs_true_raw_cosine_ci_high",
    })

    sm = teacher_sm.merge(
        gcn_plot,
        on=merge_keys,
        how="left",
        validate="one_to_one",
    )

    if sm["gcn_vs_true_raw_cosine_mean"].isna().any():
        raise ValueError("Missing GCN -> true results after merging.")
    labels = [f'{w}\n{t}' for w,t in sm[['window','target_modality']].itertuples(index=False, name=None)] if 'window' in sm.columns else sm['target_modality'].tolist()
    x = np.arange(len(sm))
    links = [
        ('Learned target -> true','teacher_vs_true_raw_cosine_mean','teacher_vs_true_raw_cosine_ci_low','teacher_vs_true_raw_cosine_ci_high'),
        ('GCN -> learned target','gcn_vs_teacher_raw_cosine_mean','gcn_vs_teacher_raw_cosine_ci_low','gcn_vs_teacher_raw_cosine_ci_high'),
        ('GCN -> true','gcn_vs_true_raw_cosine_mean','gcn_vs_true_raw_cosine_ci_low','gcn_vs_true_raw_cosine_ci_high'),
    ]
    width=0.22
    fig, ax=plt.subplots(figsize=(4.5, 3.4),dpi=600)
    fig.patch.set_alpha(0.0); ax.patch.set_alpha(0.0)
    for li,(label,mean_col,lo_col,hi_col) in enumerate(links):
        for i, r in sm.reset_index(drop=True).iterrows():
            if pd.isna(r.get(mean_col, np.nan)): continue
            p=float(r[mean_col]); lo=p-float(r[lo_col]); hi=float(r[hi_col])-p
            offset=(li-(len(links)-1)/2)*width
            ax.bar(x[i]+offset,p,width*0.92,color=CHAIN_COLORS[label],edgecolor='white',linewidth=0.5,
                   yerr=[[lo],[hi]],capsize=2,error_kw=dict(ecolor='lightgray',elinewidth=1),alpha=0.9)
            _bar_value_label(ax,x[i]+offset,p,hi)
    ax.set_xticks(x); ax.set_xticklabels(labels,fontsize=8); ax.tick_params(axis='x',length=0)
    ax.set_ylabel('Raw cosine similarity'); ax.set_title(title)
    ax.set_ylim(0, min(1.08, float(sm[[m for _,m,_,_ in links]].max().max())+0.12))
    ax.legend(handles=[Patch(facecolor=CHAIN_COLORS[l],edgecolor='white',label=l) for l,_,_,_ in links],fontsize=8,loc='upper center',bbox_to_anchor=(0.5,-0.20),frameon=False,ncol=3,handletextpad=0.6,borderaxespad=0.0)
    plt.tight_layout(); _savefig(fig,filename); plt.show()


In [ ]:
def _mean_ci(values):
    values = pd.Series(values).dropna().astype(float)
    n = len(values)
    if n == 0:
        return np.nan, np.nan, np.nan
    mean = values.mean()
    if n == 1:
        return float(mean), np.nan, np.nan
    se = values.std(ddof=1) / np.sqrt(n)
    return float(mean), float(mean - 1.96 * se), float(mean + 1.96 * se)

### Supplementary plotting functions

The following code defines the paired-trajectory and paired-difference distribution figures described above.

In [ ]:
def paired_gcn_cosine_plot(
    patient_results,
    model_a="Model A",
    model_b="Model B",
    filename="deterioration_paired_gcn_cosine.png",
):
    """Compare patient-level GCN imputation quality between models."""

    gcn = patient_results[
        patient_results["representation"] == "gcn_x0"
    ].copy()

    a = gcn[gcn["model"] == model_a][
        ["sample_id", "gcn_vs_true_raw_cosine"]
    ].rename(columns={"gcn_vs_true_raw_cosine": "model_a"})

    b = gcn[gcn["model"] == model_b][
        ["sample_id", "gcn_vs_true_raw_cosine"]
    ].rename(columns={"gcn_vs_true_raw_cosine": "model_b"})

    paired = a.merge(b, on="sample_id", how="inner")

    if paired.empty:
        raise ValueError("No paired patients found.")

    fig, ax = plt.subplots(figsize=(4, 3.4), dpi=600)
    fig.patch.set_alpha(0.0)
    ax.patch.set_alpha(0.0)

    x = np.array([0, 1])

    # Individual patient trajectories
    for _, row in paired.iterrows():
        ax.plot(
            x,
            [row["model_a"], row["model_b"]],
            color="#A0A0A0",
            alpha=0.10,
            linewidth=0.6,
            zorder=1,
        )

    # Model-level means and confidence intervals
    for j, (col, model) in enumerate([
        ("model_a", model_a),
        ("model_b", model_b),
    ]):
        mean, low, high = _mean_ci(paired[col])

        yerr = (
            None if np.isnan(low)
            else [[mean - low], [high - mean]]
        )

        ax.errorbar(
            j,
            mean,
            yerr=yerr,
            fmt="o",
            color=RUN_COLORS[model],
            ecolor="lightgray",
            capsize=4,
            markersize=4,
            linewidth=1.2,
            zorder=3,
        )

        ax.text(
            j,
            mean + (high - mean if np.isfinite(high) else 0) + 0.025,
            f"{mean:.3f}",
            ha="center",
            fontsize=8,
            color="#404040",
        )

    ax.set_xticks(x)
    ax.set_xticklabels(
        [MODEL_LABELS[model_a], MODEL_LABELS[model_b]],
        fontsize=8,
    )

    ax.set_ylabel("Raw cosine similarity")
    ax.set_title(f"MV: paired GCN imputation quality (n={len(paired)})")
    ax.tick_params(axis="x", length=0)
    ax.grid(axis="y", alpha=0.2)

    plt.tight_layout()
    _savefig(fig, filename)
    plt.show()

    return paired


def paired_gcn_cosine_delta_plot(
    patient_results,
    model_a="Model A",
    model_b="Model B",
    filename="deterioration_paired_cl_effect_distribution.png",
):
    """Plot patient-level change in GCN imputation with CL."""

    gcn = patient_results[
        patient_results["representation"] == "gcn_x0"
    ].copy()

    a = gcn[gcn["model"] == model_a][
        ["sample_id", "gcn_vs_true_raw_cosine"]
    ].rename(columns={"gcn_vs_true_raw_cosine": "no_cl"})

    b = gcn[gcn["model"] == model_b][
        ["sample_id", "gcn_vs_true_raw_cosine"]
    ].rename(columns={"gcn_vs_true_raw_cosine": "cl"})

    paired = a.merge(b, on="sample_id", how="inner")

    if paired.empty:
        raise ValueError("No paired patients found.")

    paired["delta_cl_minus_no_cl"] = (
        paired["cl"] - paired["no_cl"]
    )

    delta = paired["delta_cl_minus_no_cl"]
    mean_delta = delta.mean()
    median_delta = delta.median()

    fig, ax = plt.subplots(figsize=(5, 3.4), dpi=600)
    fig.patch.set_alpha(0.0)
    ax.patch.set_alpha(0.0)

    ax.hist(
        delta,
        bins=30,
        color=RUN_COLORS["Model B"],
        edgecolor="white",
        linewidth=0.5,
        alpha=0.9,
    )

    ax.axvline(
        0,
        color="#404040",
        linestyle="--",
        linewidth=1,
        label="No change",
    )

    ax.axvline(
        mean_delta,
        color=RUN_COLORS["Model A"],
        linewidth=1.5,
        label=f"Mean change: {mean_delta:+.3f}",
    )

    ax.set_xlabel(
        "With CL - Without CL: Δ cosine similarity"
    )
    ax.set_ylabel("Number of patients")
    ax.set_title("MV: patient-level effect of CL")

    ax.grid(axis="y", alpha=0.2)

    ax.legend(
        fontsize=8,
        loc="upper center",
        bbox_to_anchor=(0.5, -0.20),
        frameon=False,
        ncol=2,
    )

    plt.tight_layout()
    _savefig(fig, filename)
    plt.show()

    print(f"Mean paired difference: {mean_delta:.4f}")
    print(f"Median paired difference: {median_delta:.4f}")

    return paired

In [ ]:
plot_primary_imputation(
    results,
    title='MV: GCN imputation quality for CXR',
    filename='deterioration_gcn_imputation_quality.png',
)

plot_paired_cl_effect(
    results,
    title='MV: change in GCN imputation quality with CL',
    filename='deterioration_cl_effect.png',
)

plot_teacher_chain(
    results,
    title='MV: learned-target teacher-chain diagnostic',
    filename='deterioration_teacher_chain.png',
    model='Model B',
)

In [ ]:
check_raw = results[
    (results["model"] == "Model B")
    & (results["representation"] == "gcn_x0")
]

print(
    check_raw.groupby("target_modality")[
        "gcn_vs_true_raw_cosine"
    ].mean()
)

In [ ]:
# Paired GCN imputation quality
paired_results = paired_gcn_cosine_plot(results)

# Distribution of the paired CL effect
paired_delta = paired_gcn_cosine_delta_plot(results)


## 5. Interpretation and reporting limitations

These results describe **directional agreement between embeddings**, not image-pixel reconstruction or clinical predictive performance. Each model is compared against its own encoder's original embedding. The ED stay, not necessarily the patient, is the observation unit. Approximate normal-theory CIs may understate uncertainty if a patient contributes correlated stays. Paired comparisons with evaluation-order fallback IDs are valid only after verifying identical input ordering and eligibility. The learned-target chain is a diagnostic of alignment, not causal evidence about why CL helped or hurt.